# Census Block Level Roll-Up — Michigan (MI)

This notebook implements **Phase 1, Step 1.3** of the Broadband Pipeline Plan.  
It takes the raw FCC location-level CSV files for Michigan and aggregates them to the **census block level**, producing a clean dataset ready for PostgreSQL loading.

**Note:** MI current-year data contains only **Fiber to the Premises (FTTP)** records.

**Roll-up logic (per the plan):**
```
For each unique combination of (block_geoid, provider_id, holding_company, technology):
    → Take the MAX of max_advertised_download_speed
    → Take the MAX of max_advertised_upload_speed
    → Flag whether residential, business, or both are served
    → Take the MAX of low_latency
```

## 1. Import Libraries

In [1]:
import os
import pandas as pd

## 2. Load All MI CSV Files

In [2]:
### Reading all the MI CSV files into a single DataFrame
### MI current-year data is in data/MI/ — only FTTP available for J25
path = '../../data/MI/'
mi_data = pd.DataFrame()
for file in sorted(os.listdir(path)):
    if file.endswith('.csv') and file.startswith('bdc_26') and 'J25' in file:
        df = pd.read_csv(os.path.join(path, file))
        print(f"Loaded {file}: {df.shape[0]:,} rows")
        mi_data = pd.concat([mi_data, df], ignore_index=True)

print(f"Total MI records: {mi_data.shape[0]:,}")
mi_data.head()

Loaded bdc_26_FibertothePremises_fixed_broadband_J25_31mar2026.csv: 1,897,361 rows
Total MI records: 1,897,361


,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,2733897,370050,Holland Board of Public Works,1306676594,50,10000,10000,1,X,MI,261390252002005,88274841e1fffff
1,2733897,370050,Holland Board of Public Works,1306676595,50,10000,10000,1,X,MI,261390252002005,88274841e1fffff
2,2733897,370050,Holland Board of Public Works,1306676598,50,10000,10000,1,X,MI,261390252002005,88274841e1fffff
3,2733897,370050,Holland Board of Public Works,1306676600,50,10000,10000,1,X,MI,261390252002005,88274841e1fffff
4,2733897,370050,Holland Board of Public Works,1306676643,50,10000,10000,1,X,MI,261390252002005,88274841e1fffff


## 3. Basic Data Checks

In [3]:
### Data types and memory usage
print(mi_data.info())
print("
--- Missing Values ---")
print(mi_data.isnull().sum())
print("
--- Unique technology codes ---")
print(mi_data['technology'].unique())
print("
--- Unique providers ---")
print(f"{mi_data['provider_id'].nunique()} unique provider IDs")
print(f"{mi_data['brand_name'].nunique()} unique brand names")
print("
--- Business/Residential split ---")
print(mi_data['business_residential_code'].value_counts())

SyntaxError: unterminated string literal (detected at line 3) (4033020803.py, line 3)

## 4. Provider Mapping — Merge with Master Provider List

In [ ]:
### Reading providers master sheet
providers_master = pd.read_csv('../output/final_master_set.csv')
providers_master.rename(columns={'Provider ID': 'provider_id', 'FRN': 'frn'}, inplace=True)
print(f"Master provider records: {providers_master.shape[0]:,}")
providers_master.head()

In [ ]:
### Merging MI data with providers master to get holding_company
mi_data = pd.merge(mi_data, providers_master, on=['provider_id', 'frn'], how='left')

### Check for unmatched providers (NaN holding_company)
unmatched = mi_data[mi_data['holding_company'].isna()]['brand_name'].unique()
if len(unmatched) > 0:
    print(f"WARNING: {len(unmatched)} brand(s) did not match the master list:")
    print(unmatched)
    ### Fallback: use brand_name as holding_company for unmatched
    mask = mi_data['holding_company'].isna()
    mi_data.loc[mask, 'holding_company'] = mi_data.loc[mask, 'brand_name'].str.strip()
    print(f"Filled {mask.sum():,} rows with brand_name as fallback.")
else:
    print("All providers matched successfully.")

print(f"
Merged shape: {mi_data.shape}")

## 5. Technology Code Reference (Step 1.2)

| Code | Technology |
|------|------------|
| 10   | Copper Wire (DSL) |
| 40   | Cable |
| 50   | Fiber to the Premises (FTTP) |

In [ ]:
### Technology code to human-readable name mapping
TECHNOLOGY_MAP = {
    10: 'Copper Wire (DSL)',
    40: 'Cable',
    50: 'Fiber to the Premises (FTTP)'
}

### Verify all technology codes in the data are covered by the map
codes_in_data = set(mi_data['technology'].unique())
codes_in_map = set(TECHNOLOGY_MAP.keys())
unmapped = codes_in_data - codes_in_map

if unmapped:
    print(f"WARNING: Unmapped technology codes found: {unmapped}")
else:
    print(f"All technology codes covered. Codes present in data: {sorted(codes_in_data)}")

## 6. Data Cleaning — block_geoid Preparation

The `block_geoid` is a 15-digit Census Block FIPS code. We need to:
1. Drop rows with null or malformed `block_geoid`
2. Zero-pad to 15 digits (leading zeros are stripped when stored as integers)
3. Derive `state_fips` (first 2 digits), `county_fips` (first 5), and `tract_fips` (first 11)

In [ ]:
### Check for null block_geoid values
null_geoid_count = mi_data['block_geoid'].isna().sum()
print(f"Null block_geoid rows: {null_geoid_count:,}")

### Drop rows with null block_geoid
if null_geoid_count > 0:
    mi_data = mi_data.dropna(subset=['block_geoid'])
    print(f"Dropped {null_geoid_count:,} rows. Remaining: {mi_data.shape[0]:,}")

### Zero-pad block_geoid to 15 digits
mi_data['block_geoid'] = mi_data['block_geoid'].astype(int).astype(str).str.zfill(15)

### Validate: all block_geoid should be exactly 15 digits
invalid_geoid = mi_data[mi_data['block_geoid'].str.len() \!= 15]
print(f"Invalid block_geoid (not 15 digits): {invalid_geoid.shape[0]:,}")

### Derive FIPS hierarchy columns
mi_data['state_fips'] = mi_data['block_geoid'].str[:2]
mi_data['county_fips'] = mi_data['block_geoid'].str[:5]
mi_data['tract_fips'] = mi_data['block_geoid'].str[:11]

print(f"
State FIPS values: {mi_data['state_fips'].unique()}")
print(f"Unique counties: {mi_data['county_fips'].nunique()}")
print(f"Unique tracts: {mi_data['tract_fips'].nunique()}")
print(f"Unique census blocks: {mi_data['block_geoid'].nunique()}")

## 7. Census Block Level Roll-Up (Step 1.3)

Aggregate from location-level rows to census block level:
- Group by `(block_geoid, provider_id, holding_company, technology)`
- MAX of download and upload speeds
- Flag residential/business service
- MAX of low_latency

In [ ]:
### Roll up to census block level
block_level = mi_data.groupby(
    ['block_geoid', 'provider_id', 'holding_company', 'technology']
).agg(
    max_download=('max_advertised_download_speed', 'max'),
    max_upload=('max_advertised_upload_speed', 'max'),
    serves_residential=('business_residential_code', lambda x: any(v in ['R', 'X'] for v in x)),
    serves_business=('business_residential_code', lambda x: any(v in ['B', 'X'] for v in x)),
    low_latency=('low_latency', 'max'),
    state_fips=('state_fips', 'first'),
    county_fips=('county_fips', 'first'),
    tract_fips=('tract_fips', 'first')
).reset_index()

print(f"Location-level rows: {mi_data.shape[0]:,}")
print(f"Block-level rows:    {block_level.shape[0]:,}")
print(f"Reduction:           {mi_data.shape[0] / block_level.shape[0]:.1f}x")

## 8. Map Technology Codes to Names

In [ ]:
### Map technology codes to human-readable names
block_level['technology_name'] = block_level['technology'].map(TECHNOLOGY_MAP)

### Check for any unmapped codes (would result in NaN)
unmapped_rows = block_level[block_level['technology_name'].isna()]
if unmapped_rows.shape[0] > 0:
    print(f"WARNING: {unmapped_rows.shape[0]} rows with unmapped technology codes")
    print(unmapped_rows['technology'].unique())
else:
    print("All technology codes mapped successfully.")

print("
--- Technology distribution ---")
print(block_level['technology_name'].value_counts())

## 9. Reorder Columns to Match Database Schema

Target schema from the plan:
```
block_geoid, state_fips, county_fips, tract_fips,
provider_id, holding_company, technology, technology_name,
max_download, max_upload, serves_residential, serves_business, low_latency
```

In [ ]:
### Reorder columns to match the PostgreSQL block_availability table schema
block_level = block_level[[
    'block_geoid',
    'state_fips',
    'county_fips',
    'tract_fips',
    'provider_id',
    'holding_company',
    'technology',
    'technology_name',
    'max_download',
    'max_upload',
    'serves_residential',
    'serves_business',
    'low_latency'
]]

print(f"Final columns: {list(block_level.columns)}")
print(f"Final shape: {block_level.shape}")
block_level.head(10)

## 10. Validation (Step 1.4)

Sanity checks before export:
- No null `block_geoid`
- All `block_geoid` are 15 digits
- All technology codes are mapped
- Speed values are reasonable
- Residential/business flags are correct

In [ ]:
### Validation checks
print("=== VALIDATION REPORT ===")
print()

# 1. Null block_geoid
null_geoid = block_level['block_geoid'].isna().sum()
print(f"[{'PASS' if null_geoid == 0 else 'FAIL'}] Null block_geoid: {null_geoid}")

# 2. block_geoid length
bad_len = (block_level['block_geoid'].str.len() \!= 15).sum()
print(f"[{'PASS' if bad_len == 0 else 'FAIL'}] Invalid block_geoid length: {bad_len}")

# 3. Technology mapping
null_tech_name = block_level['technology_name'].isna().sum()
print(f"[{'PASS' if null_tech_name == 0 else 'FAIL'}] Unmapped technology codes: {null_tech_name}")

# 4. Speed ranges
print(f"
--- Speed ranges ---")
print(f"Download: {block_level['max_download'].min()} - {block_level['max_download'].max()} Mbps")
print(f"Upload:   {block_level['max_upload'].min()} - {block_level['max_upload'].max()} Mbps")

# 5. Residential/business flag distribution
print(f"
--- Service type flags ---")
print(f"Serves residential: {block_level['serves_residential'].sum():,} rows")
print(f"Serves business:    {block_level['serves_business'].sum():,} rows")
print(f"Both:               {(block_level['serves_residential'] & block_level['serves_business']).sum():,} rows")

# 6. Summary stats
print(f"
--- Summary ---")
print(f"Total block-level rows:  {block_level.shape[0]:,}")
print(f"Unique census blocks:    {block_level['block_geoid'].nunique():,}")
print(f"Unique providers:        {block_level['provider_id'].nunique()}")
print(f"Unique holding companies: {block_level['holding_company'].nunique()}")
print(f"Unique counties:         {block_level['county_fips'].nunique()}")
print(f"Unique tracts:           {block_level['tract_fips'].nunique()}")

In [ ]:
### Sample: pick a census block and inspect its providers/speeds
sample_block = block_level['block_geoid'].value_counts().head(1).index[0]
print(f"Sample census block: {sample_block}")
print(f"(Block with the most provider-technology combinations)
")

sample = block_level[block_level['block_geoid'] == sample_block][[
    'holding_company', 'technology_name',
    'max_download', 'max_upload', 'serves_residential', 'serves_business', 'low_latency'
]].sort_values('max_download', ascending=False)

print(sample.to_string(index=False))

## 11. Export for Database Loading

In [ ]:
### Save the block-level rollup to CSV
output_path = '../output/MI_block_level_availability.csv'
block_level.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Rows: {block_level.shape[0]:,}")
print(f"Columns: {block_level.shape[1]}")

### File size
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"File size: {file_size_mb:.1f} MB")